In [28]:
import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split

In [29]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/ailsntua/QEvasion/" + splits["train"])

In [30]:
df.head()

,title,date,president,url,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,annotator_id,annotator1,annotator2,annotator3,inaudible,multiple_questions,affirmative_questions,index,clarity_label,evasion_label
0,"The President's News Conference in Hanoi, Vietnam","September 10, 2023",Joseph R. Biden,https://www.presidency.ucsb.edu/documents/the-...,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,How would you respond to the accusation that t...,85,None,None,None,False,False,False,0,Clear Reply,Explicit
1,"The President's News Conference in Hanoi, Vietnam","September 10, 2023",Joseph R. Biden,https://www.presidency.ucsb.edu/documents/the-...,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,Do you think President Xi is being sincere abo...,85,None,None,None,False,False,False,1,Ambivalent,General
2,"The President's News Conference in Hanoi, Vietnam","September 10, 2023",Joseph R. Biden,https://www.presidency.ucsb.edu/documents/the-...,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Do you believe the country's slowdown and gro...,85,None,None,None,False,False,False,2,Ambivalent,Partial/half-answer
3,"The President's News Conference in Hanoi, Vietnam","September 10, 2023",Joseph R. Biden,https://www.presidency.ucsb.edu/documents/the-...,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Are you worried about the meeting between Pre...,85,None,None,None,False,False,False,3,Ambivalent,Dodging
4,"The President's News Conference in Hanoi, Vietnam","September 10, 2023",Joseph R. Biden,https://www.presidency.ucsb.edu/documents/the-...,3,"Q. I can imagine. It is evening, I'd like to r...","Well, I hope I get to see Mr. Xi sooner than l...",The question consists of 3 parts:\n1. Is the P...,Question part: 1. Is the President's engagemen...,Is the President's engagement with Asian coun...,85,None,None,None,False,False,False,4,Clear Reply,Explicit


In [31]:
df.columns

Index(['title', 'date', 'president', 'url', 'question_order',
       'interview_question', 'interview_answer', 'gpt3.5_summary',
       'gpt3.5_prediction', 'question', 'annotator_id', 'annotator1',
       'annotator2', 'annotator3', 'inaudible', 'multiple_questions',
       'affirmative_questions', 'index', 'clarity_label', 'evasion_label'],
      dtype='object')

In [32]:
df["question"].loc[0]

'How would you respond to the accusation that the United States is containing China while pushing for diplomatic talks?'

In [33]:
df["interview_answer"].loc[1]

"Well, look, first of all, theI am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues.And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things.And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things.And I think that one of the things we've doneI've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stability

In [34]:
def clean_text(s):
    s=re.sub(r'\.([A-Z])',r'. \1',s)#to fix words starting immediately after periods without a space.
    s=re.sub(r'([a-z])([A-Z])',r'\1 \2',s) #to fix camel case issues
    s=re.sub(r'(?<=\w)\[', ' [',s) # to add a space before brackets
    s=re.sub(r'\[\s*\]','',s) #to remove empty square brackets
    s=re.sub(r'\s+([.,!?])',r'\1',s)# to remove spaces before punctuation
    s=re.sub(r'\s{2,}',' ',s) # to remove multiple spaces
    return s.strip() #final spaces trim

In [35]:
df["interview_answer"]=df["interview_answer"].apply(clean_text) 

In [36]:
df["interview_answer"].loc[1]

"Well, look, first of all, the I am sincere about getting the relationship right. And one of the things that is going on now is, China is beginning to change some of the rules of the game, in terms of trade and other issues. And so one of the things we talked about, for example, is that they're now talking about making sure that no Chineseno one in the Chinese Government can use a Western cell phone. Those kinds of things. And so, really, what this trip was aboutit was less about containing China. I don't want to contain China. I just want to make sure that we have a relationship with China that is on the up and up, squared away, everybody knows what it's all about. And one of the ways you do that is, you make sure that we are talking about the same things. And I think that one of the things we've done I've tried to do, and I've talked with a number of my staff about this for the last, I guess, 6 monthsis, we have an opportunity to strengthen alliances around the world to maintain stab

In [37]:
#encoding task 1 labels
clarity_labels=df['clarity_label'].unique()
clarity_label_map={label: i for i, label in enumerate(clarity_labels)}
clarity_label_map

{'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}

In [38]:
df['clarity_label_id']=df['clarity_label'].map(clarity_label_map)

In [39]:
#encoding task 2 labels
evasion_labels=df['evasion_label'].unique()
evasion_label_map={label: i for i, label in enumerate(evasion_labels)}
evasion_label_map

{'Explicit': 0,
 'General': 1,
 'Partial/half-answer': 2,
 'Dodging': 3,
 'Implicit': 4,
 'Deflection': 5,
 'Declining to answer': 6,
 'Claims ignorance': 7,
 'Clarification': 8}

In [40]:
df['evasion_label_id']=df['evasion_label'].map(evasion_label_map)

In [41]:
os.makedirs("../csv_files",exist_ok=True)
df.to_csv("../csv_files/preprocessed_data.csv",index=False)

In [42]:
df["evasion_label"].value_counts()

evasion_label
Explicit               1052
Dodging                 706
Implicit                488
General                 386
Deflection              381
Declining to answer     145
Claims ignorance        119
Clarification            92
Partial/half-answer      79
Name: count, dtype: int64

In [43]:
train_df,val_df=train_test_split(df, test_size=0.2, stratify=df['evasion_label_id'], random_state=17)

In [44]:
print(f"Original shape: {df.shape}")
print(f"Train split shape: {train_df.shape}")
print(f"Validation split shape: {val_df.shape}")

Original shape: (3448, 22)
Train split shape: (2758, 22)
Validation split shape: (690, 22)


In [45]:
train_df.to_csv("../csv_files/training_data.csv",index=False)
val_df.to_csv("../csv_files/validation_data.csv",index=False)